# Building a Multi-Turn, Multi-Agent Conversational System with LangGraph

In this hands-on exercise, we’ll show you how to create a conversational application where users can interact with multiple agents, each specializing in a domain — and have a fluid, multi-turn dialogue.

You’ll learn how to:

Structure agent handoffs using

*   LangGraph's functional API
*   Use interrupts to pause for user input
*   Maintain state across multiple turns and agents.


### **Step 1: Install Required Packages**


Make sure you have the necessary libraries installed:


In [2]:
%%capture --no-stderr
%pip install -U langgraph langchain-openai

We're using:


*   langgraph for building functional workflows

*   langchain-openai to access OpenAI's models





### **Step 2: Set Your API Key**


To use OpenAI's models, you'll need to securely set your API key.

In [11]:
import getpass
import os


def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")


_set_env("GROK_API_KEY")

This will prompt you to enter your key only once.



### **Step 3: Define the Agent Tools**

Next, we’ll define some tools — these are functions the agents can call to perform specific tasks. Think of them as the agent’s capabilities.

In [12]:
import random
from typing_extensions import Literal
from langchain_core.tools import tool

@tool
def get_career_paths():
    """Suggest career options based on general user interest."""
    return random.choice(["data science", "product management", "cybersecurity"])


@tool
def get_learning_resources(career: Literal["data science", "product management", "cybersecurity"]):
    """Provide online resources or certifications for a given career path."""
    return {
        "data science": ["Coursera: IBM Data Science", "edX: Harvard's Data Science Series"],
        "product management": ["Udemy: Become a Product Manager", "Reforge Programs"],
        "cybersecurity": ["Cybrary", "CompTIA Security+ Certification"],
    }[career]


@tool(return_direct=True)
def transfer_to_education_advisor():
    """Ask education advisor agent for help."""
    return "Successfully transferred to education advisor."


@tool(return_direct=True)
def transfer_to_career_advisor():
    """Ask career advisor agent for help."""
    return "Successfully transferred to career advisor."


### **Step 4: Create the Agents**

We’ll create two specialized agents:


*   A Career Advisor who helps explore job paths
*   An Education Advisor who recommends learning resources

Each agent will use a ReAct-style decision process to decide whether to answer or hand off to the other agent.

In [14]:
_set_env("GROQ_API_KEY")

GROQ_API_KEY: ··········


In [30]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

model = ChatOpenAI(
    model="qwen/qwen3.8-27b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
    max_tokens = 400,
)

career_advisor = create_react_agent(
    model,
    tools=[get_career_paths, transfer_to_education_advisor],
    prompt=(
        "You are a career expert. Help users explore career options. "
        "If they ask about courses or education, transfer to the education advisor. "
        "Always explain your reasoning before transferring."
    ),
)

education_advisor = create_react_agent(
    model,
    tools=[get_learning_resources, transfer_to_career_advisor],
    prompt=(
        "You are an education expert. Recommend learning paths for specific careers. "
        "If the user changes their career preference, transfer back to the career advisor. "
        "Always explain your reasoning before transferring. "
        "You ARE the education advisor. Earlier messages and tool results in this "
        "conversation are your own work, so use them to answer follow-up questions. "
        "Never refer to the education advisor in the third person."
    ),
)

/tmp/ipykernel_6087/4027764113.py:11: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  career_advisor = create_react_agent(
/tmp/ipykernel_6087/4027764113.py:21: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  education_advisor = create_react_agent(


### **Step 5: Wrap the Agents in LangGraph Tasks**

Let's test a multi turn conversation with this application.

In [31]:
from langgraph.func import task

@task
def call_career_advisor(messages):
    return career_advisor.invoke({"messages": messages})["messages"]

@task
def call_education_advisor(messages):
    return education_advisor.invoke({"messages": messages})["messages"]


### **Step 6: Create the Multi-Turn Controller**

Now we’ll build the main controller — the entrypoint to your conversation graph. This handles message memory, tool routing, and waits for user input when needed.


In [32]:
from langgraph.func import entrypoint
from langgraph.graph import add_messages
from langgraph.types import interrupt, Command
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import MemorySaver
import uuid

checkpointer = MemorySaver()

def string_to_uuid(text):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, text))

@entrypoint(checkpointer=checkpointer)
def multi_turn_graph(messages, previous):
    previous = previous or []
    messages = add_messages(previous, messages)
    call_active_agent = call_career_advisor  # start with career advisor

    while True:
        agent_messages = call_active_agent(messages).result()
        messages = add_messages(messages, agent_messages)

        # Find the last AI message
        ai_msg = next(m for m in reversed(agent_messages) if isinstance(m, AIMessage))

        # If no tool was called, wait for user input
        if not ai_msg.tool_calls:
            user_input = interrupt(value="Ready for user input.")
            messages = add_messages(messages, [{
                "role": "user",
                "content": user_input,
                "id": string_to_uuid(user_input),
            }])
            continue

        # Check if the agent is transferring to the other advisor
        tool_call = ai_msg.tool_calls[-1]
        if tool_call["name"] == "transfer_to_education_advisor":
            call_active_agent = call_education_advisor
        elif tool_call["name"] == "transfer_to_career_advisor":
            call_active_agent = call_career_advisor
        else:
            raise ValueError(f"Unexpected tool: {tool_call['name']}")


### **Step 7: Test the System**

Let’s simulate a real dialogue with multiple turns. The Command(resume=...) simulates follow-up prompts from the user.

In [33]:
thread_config = {"configurable": {"thread_id": uuid.uuid4()}}

inputs = [
    {
        "role": "user",
        "content": "I'm interested in technology but not sure what career fits me.",
        "id": str(uuid.uuid4()),
    },
    Command(resume="That sounds good. What courses should I take to get started?"),
    Command(resume="Awesome! Are these resources beginner friendly?"),
]

for idx, user_input in enumerate(inputs):
    print(f"\n--- Conversation Turn {idx + 1} ---\n")
    print(f"User: {user_input}\n")
    for update in multi_turn_graph.stream(
        user_input,
        config=thread_config,
        stream_mode="updates",
    ):
        for node_id, value in update.items():
            if isinstance(value, list) and value:
                last_msg = value[-1]
                if isinstance(last_msg, dict) or last_msg.type != "ai":
                    continue
                print(f"{node_id}: {last_msg.content}")

print("LAYA successfully completed running this langgraph demo")



--- Conversation Turn 1 ---

User: {'role': 'user', 'content': "I'm interested in technology but not sure what career fits me.", 'id': '0388d435-701e-4f97-8f7e-147d0ef730a6'}

call_career_advisor: Based on the output, **Product Management** is a very strong and highly recommended career option for you.

Here’s why it might be a great fit, even if you’re not a hardcore developer:

1. **The Bridge Role:** Product Managers sit at the intersection of **Business, Technology, and Design**. If you enjoy technology but also have an interest in strategy, leadership, or solving user problems, this is the ideal role.
2. **Scope & Impact:** Instead of just coding a specific feature, you define *what* gets built and *why*. You manage the product roadmap, work with engineers to estimate timelines, and partner with designers to create user-centric solutions.
3. **Demand for Diverse Skills:** While a technical background is a huge plus, you don't always need it. The most successful PMs are often thos

In [20]:
import requests, os

r = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
)
for m in r.json()["data"]:
    print(m["id"])


whisper-large-v3-turbo
canopylabs/orpheus-v1-english
canopylabs/orpheus-arabic-saudi
whisper-large-v3
openai/gpt-oss-safeguard-20b
allam-2-7b
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-120b
qwen/qwen3.8-27b
openai/gpt-oss-20b
